# Reproduce lowbit-atlas on this machine

This notebook rebuilds runtimes and reruns measurements of [lowbit-atlas](https://github.com/matheus-c-feitosa/lowbit-atlas) from scratch, then compares them with the published numbers: `SAME`, `CLOSE`, `DIFFERS` or `NEW`.

- **Short check, the default (`MODE=check`, about 30 min on a 4-vCPU Kaggle VM):** builds Prism and ik_llama.cpp and checks Bonsai 1.7B (Q1_0, PQ2_0) and BitNet b1.58 2B4T: greedy text, perplexity (8 × 512) and speed.
- **Full run (`MODE=quick`, about 40 min of builds and 3 h of measurements):** everything in the report except the 27B model. Set `MODE=quick` at the top of the cell.
- **Kaggle:** turn on **Internet** in the notebook settings (it needs a phone-verified account). A CPU session is enough; no GPU is used.
- **Colab:** works on the free CPU runtime; the short check takes about 1 h on its 2 vCPUs.
- **Most useful:** a CPU class that is not measured yet, such as AVX-512, AVX-VNNI or Arm with i8mm/SVE. Its numbers come out as `NEW`.

When it finishes, please share the summary it prints: [open a results issue](https://github.com/matheus-c-feitosa/lowbit-atlas/issues/new?template=results.yml) and paste it there. New CPU classes go into the report, with credit.


In [ ]:
%%bash
# lowbit-atlas: build the runtimes, run the measurements, compare.
# Same steps as the Reproduce section of the README. Linux only.
set -u
W=${W:-$PWD/lowbit-run}
REPO=${REPO:-https://github.com/matheus-c-feitosa/lowbit-atlas}
MODE=${MODE:-check}  # check: ~30 min | quick: ~3.5 h | full: + the 27B
# check needs only Prism and ik_llama.cpp; quick and full need all runtimes
if [ "$MODE" = check ]; then TARGETS=${TARGETS:-prism ik}
else TARGETS=${TARGETS:-all}; fi
mkdir -p "$W" && cd "$W" || { echo "cannot use $W"; exit 1; }
S=$W/SUMMARY.txt; : > "$S"
log(){ echo "$*" | tee -a "$S"; }
fin(){
  tar czf "$W/results.tar.gz" -C "$W" SUMMARY.txt compare.txt \
    work/results.jsonl work/logs build.log run.log 2>/dev/null
  echo; echo "===== SUMMARY (paste this in the results issue) ====="
  cat "$S"; echo "Full logs: $W/results.tar.gz"; }
fail(){ log "STOPPED: $*"; fin; exit 1; }
log "cpu: $(lscpu | sed -n 's/^Model name: *//p' | head -1)" \
  "| $(uname -m) | $(nproc) vCPUs | $(free -g | awk '/Mem/{print $2}') GB"
log "flags: $(grep -o -w -E 'avx2|avx512f|avx512_vnni|avx_vnni|amx_tile|fma|f16c|asimddp|i8mm|sve|sve2' \
  /proc/cpuinfo | sort -u | tr '\n' ' ')"
log "compiler: $(gcc --version | head -1)"
rm -rf repo
git clone -q --depth 1 "$REPO" repo < /dev/null || fail "git clone $REPO"
log "repo: $REPO @ $(git -C repo rev-parse --short HEAD)"
export WORK=$W/work JOBS=$(nproc) THREADS=${THREADS:-2}
mkdir -p "$WORK"
t0=$SECONDS
repo/scripts/build.sh $TARGETS < /dev/null > build.log 2>&1 \
  || fail "build.sh: $(tail -2 build.log | tr '\n' ' ')"
log "build.sh $TARGETS: $((SECONDS - t0)) s"
t0=$SECONDS
repo/scripts/run.sh "$MODE" < /dev/null > run.log 2>&1 \
  || fail "run.sh: $(tail -2 run.log | tr '\n' ' ')"
log "run.sh $MODE (THREADS=$THREADS): $((SECONDS - t0)) s"
repo/scripts/compare.py "$WORK/results.jsonl" > compare.txt 2>&1
log "compare.py exit code: $? (1 means at least one DIFFERS)"
grep -v '^INFO' compare.txt | cut -c1-150 | tee -a "$S" > /dev/null
grep '^INFO' compare.txt | cut -c1-110 | tee -a "$S" > /dev/null
fin


**Reading the summary:** `SAME` means identical to the published value; `CLOSE` means within the spread between valid code paths (0.2% PPL, 2× KLD); `DIFFERS` is worth a look; `NEW` means nothing is published yet for this label on this CPU architecture. Speed lines (`INFO`) are reported, never judged.
